# Notebook 20: Regional Analysis & SH Encoding Study

**Date**: 2026-01-12  
**Goal**: Test if regional/local scale or SH encoding dimensionality reveals spline advantages

**Motivation**: NB19/19b found NO spline advantage at global scale with SH(L=10). Test if:
1. **Continental differences** (mountainous vs flat terrain)
2. **SH encoding dimensionality** (L=10 vs L=20 vs L=40)

reveal patterns masked in global aggregation.

---

## Experiments in This Notebook

### Phase 1 (This Session - Existing Data)
- ✅ **Experiment 1**: Continental Comparisons (5 continents)
- ✅ **Experiment 2**: SH Encoding Levels (L=10/20/40)

### Phase 2 (Future - New Data)
- ⏳ **Experiment 3**: Multi-Resolution Within Regions (SRTM 30m)
- ⏳ **Experiment 4**: Urban vs Rural Patterns
- ⏳ **Experiment 5**: Boundary-Rich Tasks (Coastlines, Land Cover)

---

## Expected Runtime
- Exp 1: ~4 hours (5 regions × 3 activations)
- Exp 2: ~8 hours (3 SH levels × 2 terrains × 3 activations)
- **Total Phase 1**: ~12 hours on Colab T4 GPU

---
## Setup

In [1]:
# Environment setup
import os
import sys

if 'COLAB_GPU' in os.environ:
    !rm -rf sample_data .config satclip 2>/dev/null
    !git clone https://github.com/1hamzaiqbal/satclip.git
    !pip install lightning torchgeo huggingface_hub rasterio --quiet
    sys.path.append('./satclip/satclip')
else:
    sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'satclip'))

Cloning into 'satclip'...
remote: Enumerating objects: 606, done.
remote: Counting objects: 100% (285/285), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 606 (delta 208), reused 203 (delta 159), pack-reused 321 (from 2)
Receiving objects: 100% (606/606), 82.83 MiB | 21.64 MiB/s, done.
Resolving deltas: 100% (311/311), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.0/846.0 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 650.7/650.7 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.9/243.9 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.3/859.3 kB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.5/849.5 kB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 17.5 MB/s eta 0:00:00
   ━━━

In [2]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import r2_score

import positional_encoding as PE
import xarray as xr

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


---
## Model Definitions (From NB19)

In [3]:
class SplineActivation(nn.Module):
    def __init__(self, n_knots=15, input_range=(-3.0, 3.0), init='relu'):
        super().__init__()
        self.n_knots = n_knots
        self.input_range = input_range
        knot_x = torch.linspace(input_range[0], input_range[1], n_knots)
        self.register_buffer('knot_x', knot_x)
        if init == 'relu':
            knot_y = torch.relu(knot_x)
        elif init == 'linear':
            knot_y = knot_x.clone()
        else:
            knot_y = torch.randn(n_knots) * 0.1
        self.knot_y = nn.Parameter(knot_y)

    def forward(self, x):
        x_clamped = torch.clamp(x, self.input_range[0], self.input_range[1])
        x_norm = (x_clamped - self.knot_x[0]) / (self.knot_x[-1] - self.knot_x[0])
        x_idx = x_norm * (self.n_knots - 1)
        idx_low = torch.floor(x_idx).long()
        idx_high = torch.clamp(idx_low + 1, max=self.n_knots - 1)
        idx_low = torch.clamp(idx_low, max=self.n_knots - 1)
        weight = x_idx - idx_low.float()
        y_low = self.knot_y[idx_low]
        y_high = self.knot_y[idx_high]
        return y_low + weight * (y_high - y_low)


class SirenLayer(nn.Module):
    def __init__(self, dim_in, dim_out, w0=1.0, is_first=False):
        super().__init__()
        self.dim_in = dim_in
        self.w0 = w0
        self.is_first = is_first
        self.linear = nn.Linear(dim_in, dim_out)
        self._init_weights()

    def _init_weights(self):
        if self.is_first:
            bound = 1.0 / self.dim_in
        else:
            bound = np.sqrt(6.0 / self.dim_in) / self.w0
        self.linear.weight.data.uniform_(-bound, bound)
        if self.linear.bias is not None:
            self.linear.bias.data.uniform_(-bound, bound)

    def forward(self, x):
        return torch.sin(self.w0 * self.linear(x))


class UniversalEncoder(nn.Module):
    def __init__(self, input_type='sh', sh_legendre_polys=10,
                 activation_type='spline', activation_kwargs=None,
                 n_layers=3, hidden_dim=256, output_dim=256):
        super().__init__()
        self.input_type = input_type
        self.activation_type = activation_type

        if input_type == 'raw':
            self.posenc = None
            input_dim = 2
        elif input_type == 'sh':
            self.posenc = PE.SphericalHarmonics(
                legendre_polys=sh_legendre_polys,
                harmonics_calculation='analytic'
            )
            with torch.no_grad():
                test_coords = torch.zeros(1, 2)
                test_output = self.posenc(test_coords)
                input_dim = test_output.shape[1]
        else:
            raise ValueError(f"Unknown input_type: {input_type}")

        if activation_kwargs is None:
            activation_kwargs = {}

        dims = [input_dim] + [hidden_dim] * n_layers + [output_dim]

        if activation_type == 'siren':
            self.layers = nn.ModuleList()
            for i in range(len(dims) - 1):
                is_first = (i == 0)
                w0 = 30.0 if is_first else 1.0
                if i < len(dims) - 2:
                    self.layers.append(SirenLayer(dims[i], dims[i+1], w0=w0, is_first=is_first))
                else:
                    linear = nn.Linear(dims[i], dims[i+1])
                    bound = np.sqrt(6.0 / dims[i]) / 1.0
                    linear.weight.data.uniform_(-bound, bound)
                    if linear.bias is not None:
                        linear.bias.data.uniform_(-bound, bound)
                    self.layers.append(linear)
            self.activations = None

        elif activation_type == 'spline':
            self.linears = nn.ModuleList([
                nn.Linear(dims[i], dims[i+1])
                for i in range(len(dims) - 1)
            ])
            self.activations = nn.ModuleList([
                SplineActivation(**activation_kwargs)
                for _ in range(n_layers)
            ])
            for linear in self.linears:
                nn.init.kaiming_normal_(linear.weight)
                nn.init.zeros_(linear.bias)

        elif activation_type == 'relu':
            layers = []
            for i in range(len(dims) - 1):
                layers.append(nn.Linear(dims[i], dims[i+1]))
                if i < len(dims) - 2:
                    layers.append(nn.ReLU())
            self.net = nn.Sequential(*layers)
            for m in self.modules():
                if isinstance(m, nn.Linear):
                    nn.init.kaiming_normal_(m.weight)
                    nn.init.zeros_(m.bias)
        else:
            raise ValueError(f"Unknown activation_type: {activation_type}")

    def forward(self, coords):
        if self.input_type == 'raw':
            x = coords / torch.tensor([180., 90.], device=coords.device)
        else:
            x = self.posenc(coords)

        if self.activation_type == 'siren':
            for layer in self.layers:
                x = layer(x)
        elif self.activation_type == 'spline':
            for i, (linear, act) in enumerate(zip(self.linears[:-1], self.activations)):
                x = act(linear(x))
            x = self.linears[-1](x)
        else:
            x = self.net(x)

        return x


class RegressionPredictor(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 1)
        )

    def forward(self, coords):
        return self.head(self.encoder(coords)).squeeze(-1)


print("✅ Model classes loaded")

✅ Model classes loaded


---
## Training Utilities (From NB19)

In [4]:
def sample_regional_blocked(data, lons, lats, region_bounds, n_samples=5000,
                           grid_size=5.0, test_ratio=0.3, seed=42):
    """
    Sample from a regional subset with spatial blocking.

    region_bounds: dict with 'lat_min', 'lat_max', 'lon_min', 'lon_max'
    """
    np.random.seed(seed)

    # Find indices for region
    lat_mask = (lats >= region_bounds['lat_min']) & (lats <= region_bounds['lat_max'])
    lon_mask = (lons >= region_bounds['lon_min']) & (lons <= region_bounds['lon_max'])

    # Get lat/lon indices
    lat_idx = np.where(lat_mask)[0]
    lon_idx = np.where(lon_mask)[0]

    # Extract regional data
    regional_data = data[np.ix_(lat_idx, lon_idx)]
    regional_lats = lats[lat_idx]
    regional_lons = lons[lon_idx]

    # Create meshgrid
    lon_grid, lat_grid = np.meshgrid(regional_lons, regional_lats)

    # Valid data mask
    valid = regional_data > -1e30

    # Flatten
    valid_lons = lon_grid[valid]
    valid_lats = lat_grid[valid]
    valid_vals = regional_data[valid]

    # Sample subset
    n_valid = len(valid_vals)
    if n_valid > n_samples:
        sample_idx = np.random.choice(n_valid, n_samples, replace=False)
        sample_lons = valid_lons[sample_idx]
        sample_lats = valid_lats[sample_idx]
        sample_vals = valid_vals[sample_idx]
    else:
        sample_lons = valid_lons
        sample_lats = valid_lats
        sample_vals = valid_vals

    # Spatial blocking for train/test split
    region_width = region_bounds['lon_max'] - region_bounds['lon_min']
    region_height = region_bounds['lat_max'] - region_bounds['lat_min']

    n_lon_cells = max(1, int(region_width / grid_size))
    n_lat_cells = max(1, int(region_height / grid_size))
    n_cells = n_lon_cells * n_lat_cells

    test_cells = set(np.random.choice(n_cells, max(1, int(n_cells * test_ratio)), replace=False))

    train_mask = []
    for lon, lat in zip(sample_lons, sample_lats):
        lon_cell = int((lon - region_bounds['lon_min']) / grid_size)
        lat_cell = int((lat - region_bounds['lat_min']) / grid_size)
        lon_cell = min(lon_cell, n_lon_cells - 1)
        lat_cell = min(lat_cell, n_lat_cells - 1)
        cell = lat_cell * n_lon_cells + lon_cell
        train_mask.append(cell not in test_cells)

    train_mask = np.array(train_mask)
    coords = np.stack([sample_lons, sample_lats], axis=1)

    return coords[train_mask], sample_vals[train_mask], coords[~train_mask], sample_vals[~train_mask]


def train_elevation_model(name, encoder, coords_train, vals_train, coords_test, vals_test,
                         epochs=100, batch_size=256, lr=1e-3, verbose=False):
    """
    Train elevation prediction model.
    """
    model = RegressionPredictor(encoder).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    # Normalize: z-score then shift for log1p
    train_mean, train_std = vals_train.mean(), vals_train.std()
    vals_train_norm = (vals_train - train_mean) / train_std
    vals_test_norm = (vals_test - train_mean) / train_std

    # Shift to make all positive for log1p
    shift = -vals_train_norm.min() + 1
    vals_train_shifted = vals_train_norm + shift
    vals_test_shifted = vals_test_norm + shift

    train_X = torch.tensor(coords_train, dtype=torch.float32)
    train_y = torch.tensor(np.log1p(vals_train_shifted), dtype=torch.float32)
    test_X = torch.tensor(coords_test, dtype=torch.float32).to(device)
    test_y = torch.tensor(np.log1p(vals_test_shifted), dtype=torch.float32)

    loader = DataLoader(TensorDataset(train_X, train_y),
                       batch_size=batch_size, shuffle=True)

    best_r2 = -float('inf')
    start = time.time()

    for epoch in range(epochs):
        model.train()
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            loss = loss_fn(model(X), y)
            loss.backward()
            opt.step()

        if (epoch + 1) % 20 == 0 or epoch == 0:
            model.eval()
            with torch.no_grad():
                pred = model(test_X).cpu().numpy()
            r2 = r2_score(test_y.numpy(), pred)
            best_r2 = max(best_r2, r2)
            if verbose:
                print(f"  Epoch {epoch+1}/{epochs}: R² = {r2:.4f}")

    train_time = time.time() - start
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return {
        'model': name,
        'r2': best_r2,
        'params': n_params,
        'time': train_time,
    }


print("✅ Training utilities loaded")

✅ Training utilities loaded


---
## Load Data

In [5]:
print("="*70)
print("LOADING ETOPO ELEVATION DATA")
print("="*70)

# Check if running on Colab or local
if 'COLAB_GPU' in os.environ:
    # On Colab - need to download
    !wget -q -O etopo_60s.nc "https://www.ngdc.noaa.gov/thredds/fileServer/global/ETOPO2022/60s/60s_surface_elev_netcdf/ETOPO_2022_v1_60s_N90W180_surface.nc"
    data_path = 'etopo_60s.nc'
else:
    # Local - check if file exists
    data_path = 'etopo_60s.nc'
    if not os.path.exists(data_path):
        print(f"❌ Data file not found: {data_path}")
        print("Please download from NB19 or run data acquisition script")

# Load with xarray
ds = xr.open_dataset(data_path)
elevation = ds['z'].values
lats = ds['lat'].values
lons = ds['lon'].values

print(f"✅ Elevation data loaded: {elevation.shape}")
print(f"   Latitude range: [{lats.min():.2f}, {lats.max():.2f}]")
print(f"   Longitude range: [{lons.min():.2f}, {lons.max():.2f}]")
print(f"   Elevation range: [{elevation.min():.2f}, {elevation.max():.2f}] meters")
print("="*70)

LOADING ETOPO ELEVATION DATA
✅ Elevation data loaded: (10800, 21600)
   Latitude range: [-89.99, 89.99]
   Longitude range: [-179.99, 179.99]
   Elevation range: [-10752.08, 8157.36] meters


---
## Experiment 1: Continental Comparisons

**Hypothesis**: Mountainous regions favor splines, flat regions favor ReLU

**Design**: 5 continents × 3 activations (ReLU, Spline, SIREN)

In [6]:
# Define continental regions
REGIONS = {
    'north_america': {
        'lat_min': 25, 'lat_max': 50,
        'lon_min': -125, 'lon_max': -65,
        'terrain': 'mixed',  # Rockies + plains
        'description': 'Rockies (west) + Great Plains (east)'
    },
    'europe': {
        'lat_min': 35, 'lat_max': 65,
        'lon_min': -10, 'lon_max': 40,
        'terrain': 'mixed',  # Alps + plains
        'description': 'Alps (south) + European Plain (north)'
    },
    'asia_himalayas': {
        'lat_min': 25, 'lat_max': 40,
        'lon_min': 70, 'lon_max': 100,
        'terrain': 'mountainous',  # Himalayas
        'description': 'Himalayas - highest relief on Earth'
    },
    'africa_sahara': {
        'lat_min': 15, 'lat_max': 30,
        'lon_min': -10, 'lon_max': 30,
        'terrain': 'flat',  # Sahara Desert
        'description': 'Sahara Desert - low relief'
    },
    'south_america_andes': {
        'lat_min': -40, 'lat_max': 10,
        'lon_min': -80, 'lon_max': -60,
        'terrain': 'mountainous',  # Andes
        'description': 'Andes Mountains - high relief'
    },
}

# Display regions
print("Continental Regions:")
for name, bounds in REGIONS.items():
    print(f"  {name:20s}: {bounds['description']:40s} [Terrain: {bounds['terrain']}]")

Continental Regions:
  north_america       : Rockies (west) + Great Plains (east)     [Terrain: mixed]
  europe              : Alps (south) + European Plain (north)    [Terrain: mixed]
  asia_himalayas      : Himalayas - highest relief on Earth      [Terrain: mountainous]
  africa_sahara       : Sahara Desert - low relief               [Terrain: flat]
  south_america_andes : Andes Mountains - high relief            [Terrain: mountainous]


In [7]:
print("="*80)
print("EXPERIMENT 1: CONTINENTAL COMPARISONS")
print("="*80)

results_exp1 = []
activations = ['relu', 'spline', 'siren']

for region_name, region_bounds in REGIONS.items():
    print(f"\n{'='*80}")
    print(f"Region: {region_name.upper().replace('_', ' ')}")
    print(f"Terrain: {region_bounds['terrain']}")
    print(f"Bounds: Lat [{region_bounds['lat_min']}, {region_bounds['lat_max']}], "
          f"Lon [{region_bounds['lon_min']}, {region_bounds['lon_max']}]")
    print("="*80)

    # Sample regional data
    try:
        coords_train, vals_train, coords_test, vals_test = sample_regional_blocked(
            elevation, lons, lats, region_bounds, n_samples=5000
        )
        print(f"Samples: {len(coords_train)} train, {len(coords_test)} test")
    except Exception as e:
        print(f"❌ Error sampling region: {e}")
        continue

    # Test each activation
    for act in activations:
        print(f"\n--- Testing {act.upper()} ---")

        if act == 'spline':
            kwargs = {'n_knots': 15, 'init': 'relu'}
        else:
            kwargs = None

        enc = UniversalEncoder(
            input_type='sh',
            sh_legendre_polys=10,  # L=10 baseline
            activation_type=act,
            activation_kwargs=kwargs
        )

        res = train_elevation_model(
            f'{region_name}_{act}',
            enc,
            coords_train, vals_train,
            coords_test, vals_test,
            verbose=True
        )

        res['region'] = region_name
        res['terrain'] = region_bounds['terrain']
        res['activation'] = act
        res['sh_level'] = 10

        results_exp1.append(res)
        print(f"Final R²: {res['r2']:.4f}, Time: {res['time']:.1f}s")

# Convert to DataFrame
df_exp1 = pd.DataFrame(results_exp1)

print("\n" + "="*80)
print("EXPERIMENT 1 SUMMARY")
print("="*80)
print(df_exp1[['region', 'terrain', 'activation', 'r2', 'time']].to_string(index=False))
print("="*80)

EXPERIMENT 1: CONTINENTAL COMPARISONS

Region: NORTH AMERICA
Terrain: mixed
Bounds: Lat [25, 50], Lon [-125, -65]
Samples: 3489 train, 1511 test

--- Testing RELU ---
  Epoch 1/100: R² = 0.1442
  Epoch 20/100: R² = 0.9063
  Epoch 40/100: R² = 0.8858
  Epoch 60/100: R² = 0.8853
  Epoch 80/100: R² = 0.8776
  Epoch 100/100: R² = 0.8668
Final R²: 0.9063, Time: 25.7s

--- Testing SPLINE ---
  Epoch 1/100: R² = 0.0350
  Epoch 20/100: R² = 0.8961
  Epoch 40/100: R² = 0.9094
  Epoch 60/100: R² = 0.9224
  Epoch 80/100: R² = 0.9109
  Epoch 100/100: R² = 0.9224
Final R²: 0.9224, Time: 40.3s

--- Testing SIREN ---
  Epoch 1/100: R² = 0.6564
  Epoch 20/100: R² = 0.8480
  Epoch 40/100: R² = 0.8773
  Epoch 60/100: R² = 0.9077
  Epoch 80/100: R² = 0.9096
  Epoch 100/100: R² = 0.9340
Final R²: 0.9340, Time: 25.2s

Region: EUROPE
Terrain: mixed
Bounds: Lat [35, 65], Lon [-10, 40]
Samples: 3494 train, 1506 test

--- Testing RELU ---
  Epoch 1/100: R² = 0.0855
  Epoch 20/100: R² = 0.4352
  Epoch 40/100: R

In [8]:
# Analyze results by terrain type
print("\n" + "="*80)
print("ANALYSIS: BY TERRAIN TYPE")
print("="*80)

for terrain in ['mountainous', 'flat', 'mixed']:
    terrain_data = df_exp1[df_exp1['terrain'] == terrain]

    if len(terrain_data) == 0:
        continue

    print(f"\n{terrain.upper()}:")

    # Pivot to compare activations
    pivot = terrain_data.pivot_table(values='r2', index='region', columns='activation')

    if 'relu' in pivot.columns and 'spline' in pivot.columns:
        pivot['advantage'] = pivot['spline'] - pivot['relu']
        pivot['advantage_pct'] = 100 * pivot['advantage'] / pivot['relu']

    print(pivot.to_string())

    # Average advantage for this terrain
    if 'advantage_pct' in pivot.columns:
        avg_adv = pivot['advantage_pct'].mean()
        print(f"\nAverage Spline Advantage: {avg_adv:+.2f}%")

print("\n" + "="*80)

# Save results
df_exp1.to_csv('exp1_continental_comparisons.csv', index=False)
print("\n✅ Results saved to exp1_continental_comparisons.csv")


ANALYSIS: BY TERRAIN TYPE

MOUNTAINOUS:
activation               relu     siren    spline  advantage  advantage_pct
region                                                                     
asia_himalayas       0.656462  0.506104  0.602509  -0.053953      -8.218693
south_america_andes  0.848198  0.785430  0.826314  -0.021885      -2.580123

Average Spline Advantage: -5.40%

FLAT:
activation         relu     siren   spline  advantage  advantage_pct
region                                                              
africa_sahara  0.699049  0.807889  0.79254   0.093491      13.373976

Average Spline Advantage: +13.37%

MIXED:
activation         relu     siren    spline  advantage  advantage_pct
region                                                               
europe         0.445076  0.326873  0.431748  -0.013329      -2.994681
north_america  0.906255  0.933980  0.922425   0.016170       1.784249

Average Spline Advantage: -0.61%


✅ Results saved to exp1_continental_comparisons.

---
## Experiment 2: SH Encoding Levels

**Hypothesis**: Higher SH dimensionality (L=20, L=40) reveals spline advantages

**Design**: 3 SH levels × 2 terrains (mountain vs flat) × 3 activations

In [ ]:
print("="*80)
print("EXPERIMENT 2: SH ENCODING LEVELS")
print("="*80)

# Select representative regions for each terrain type
TEST_REGIONS = {
    'mountain': 'asia_himalayas',  # Strongest mountainous region
    'flat': 'africa_sahara',       # Flattest region
}

SH_LEVELS = [10, 20, 40]
results_exp2 = []

for terrain_type, region_name in TEST_REGIONS.items():
    region_bounds = REGIONS[region_name]

    print(f"\n{'='*80}")
    print(f"Terrain: {terrain_type.upper()} ({region_name})")
    print("="*80)

    # Sample region once
    coords_train, vals_train, coords_test, vals_test = sample_regional_blocked(
        elevation, lons, lats, region_bounds, n_samples=5000
    )
    print(f"Samples: {len(coords_train)} train, {len(coords_test)} test")

    for sh_level in SH_LEVELS:
        print(f"\n--- SH Level L={sh_level} ({(sh_level+1)**2} dimensions) ---")

        for act in activations:
            print(f"  Testing {act.upper()}...", end=" ")

            if act == 'spline':
                kwargs = {'n_knots': 15, 'init': 'relu'}
            else:
                kwargs = None

            # Adjust hidden_dim for L=40 to avoid memory issues
            if sh_level == 40:
                hidden_dim = 128  # Reduce from 256
            else:
                hidden_dim = 256

            enc = UniversalEncoder(
                input_type='sh',
                sh_legendre_polys=sh_level,
                activation_type=act,
                activation_kwargs=kwargs,
                hidden_dim=hidden_dim
            )

            res = train_elevation_model(
                f'{terrain_type}_L{sh_level}_{act}',
                enc,
                coords_train, vals_train,
                coords_test, vals_test,
                verbose=False
            )

            res['terrain'] = terrain_type
            res['region'] = region_name
            res['activation'] = act
            res['sh_level'] = sh_level
            res['sh_dims'] = (sh_level + 1) ** 2
            res['hidden_dim'] = hidden_dim

            results_exp2.append(res)
            print(f"R²: {res['r2']:.4f}, Time: {res['time']:.1f}s")

# Convert to DataFrame
df_exp2 = pd.DataFrame(results_exp2)

print("\n" + "="*80)
print("EXPERIMENT 2 SUMMARY")
print("="*80)
print(df_exp2[['terrain', 'sh_level', 'activation', 'r2', 'time']].to_string(index=False))
print("="*80)

In [ ]:
# Analyze SH level effects
print("\n" + "="*80)
print("ANALYSIS: SH ENCODING LEVEL EFFECTS")
print("="*80)

for terrain in ['mountain', 'flat']:
    terrain_data = df_exp2[df_exp2['terrain'] == terrain]

    print(f"\n{terrain.upper()}:")

    # Pivot: SH levels vs activations
    pivot = terrain_data.pivot_table(values='r2', index='sh_level', columns='activation')

    if 'relu' in pivot.columns and 'spline' in pivot.columns:
        pivot['advantage'] = pivot['spline'] - pivot['relu']
        pivot['advantage_pct'] = 100 * pivot['advantage'] / pivot['relu']

    print(pivot.to_string())

    # Check if advantage increases with L
    if 'advantage_pct' in pivot.columns:
        print(f"\nAdvantage Trend:")
        for sh_level in SH_LEVELS:
            if sh_level in pivot.index:
                adv = pivot.loc[sh_level, 'advantage_pct']
                print(f"  L={sh_level:2d}: {adv:+.2f}%")

print("\n" + "="*80)

# Save results
df_exp2.to_csv('exp2_sh_encoding_levels.csv', index=False)
print("\n✅ Results saved to exp2_sh_encoding_levels.csv")

---
## Combined Analysis

Synthesize findings from both experiments

In [ ]:
print("="*80)
print("PHASE 1 SYNTHESIS: EXPERIMENTS 1 & 2")
print("="*80)

# Key Question 1: Does terrain matter?
print("\n🎯 KEY QUESTION 1: Does terrain type matter?\n")

if len(df_exp1) > 0:
    # Compare mountainous vs flat at L=10
    mountain_data = df_exp1[df_exp1['terrain'] == 'mountainous']
    flat_data = df_exp1[df_exp1['terrain'] == 'flat']

    if len(mountain_data) > 0 and len(flat_data) > 0:
        # Spline advantage in each terrain
        mountain_spline = mountain_data[mountain_data['activation'] == 'spline']['r2'].mean()
        mountain_relu = mountain_data[mountain_data['activation'] == 'relu']['r2'].mean()
        mountain_adv = 100 * (mountain_spline - mountain_relu) / mountain_relu

        flat_spline = flat_data[flat_data['activation'] == 'spline']['r2'].mean()
        flat_relu = flat_data[flat_data['activation'] == 'relu']['r2'].mean()
        flat_adv = 100 * (flat_spline - flat_relu) / flat_relu

        print(f"Mountainous regions: Spline advantage = {mountain_adv:+.2f}%")
        print(f"Flat regions:        Spline advantage = {flat_adv:+.2f}%")
        print(f"Difference:          {mountain_adv - flat_adv:+.2f} percentage points")

        if abs(mountain_adv - flat_adv) > 1.0:
            print("\n✅ TERRAIN MATTERS: >1% difference between mountain and flat")
        else:
            print("\n❌ TERRAIN DOESN'T MATTER: <1% difference")

# Key Question 2: Does SH level matter?
print("\n" + "-"*80)
print("🎯 KEY QUESTION 2: Does SH encoding dimensionality matter?\n")

if len(df_exp2) > 0:
    # For each terrain, check if L=40 > L=10
    for terrain in ['mountain', 'flat']:
        terrain_data = df_exp2[df_exp2['terrain'] == terrain]

        if len(terrain_data) > 0:
            # Spline advantage at each SH level
            print(f"{terrain.upper()}:")
            for sh_level in [10, 20, 40]:
                level_data = terrain_data[terrain_data['sh_level'] == sh_level]
                if len(level_data) > 0:
                    spline_r2 = level_data[level_data['activation'] == 'spline']['r2'].values[0]
                    relu_r2 = level_data[level_data['activation'] == 'relu']['r2'].values[0]
                    adv = 100 * (spline_r2 - relu_r2) / relu_r2
                    print(f"  L={sh_level:2d}: Spline advantage = {adv:+.2f}%")
            print()

    # Check if L=40 shows consistent improvement
    l10_data = df_exp2[df_exp2['sh_level'] == 10]
    l40_data = df_exp2[df_exp2['sh_level'] == 40]

    if len(l10_data) > 0 and len(l40_data) > 0:
        l10_spline_adv = []
        l40_spline_adv = []

        for terrain in ['mountain', 'flat']:
            l10_terrain = l10_data[l10_data['terrain'] == terrain]
            l40_terrain = l40_data[l40_data['terrain'] == terrain]

            if len(l10_terrain) > 0 and len(l40_terrain) > 0:
                l10_s = l10_terrain[l10_terrain['activation'] == 'spline']['r2'].values[0]
                l10_r = l10_terrain[l10_terrain['activation'] == 'relu']['r2'].values[0]
                l10_spline_adv.append(100 * (l10_s - l10_r) / l10_r)

                l40_s = l40_terrain[l40_terrain['activation'] == 'spline']['r2'].values[0]
                l40_r = l40_terrain[l40_terrain['activation'] == 'relu']['r2'].values[0]
                l40_spline_adv.append(100 * (l40_s - l40_r) / l40_r)

        if len(l10_spline_adv) > 0 and len(l40_spline_adv) > 0:
            avg_l10 = np.mean(l10_spline_adv)
            avg_l40 = np.mean(l40_spline_adv)
            improvement = avg_l40 - avg_l10

            print(f"Average spline advantage at L=10: {avg_l10:+.2f}%")
            print(f"Average spline advantage at L=40: {avg_l40:+.2f}%")
            print(f"Improvement from L=10 to L=40:    {improvement:+.2f} percentage points")

            if improvement > 1.0:
                print("\n✅ SH LEVEL MATTERS: L=40 shows >1% better spline advantage")
            else:
                print("\n❌ SH LEVEL DOESN'T MATTER: <1% improvement from L=10 to L=40")

print("\n" + "="*80)

In [ ]:
# Recommendations for next steps
print("="*80)
print("NEXT STEPS DECISION")
print("="*80)

print("\nBased on Phase 1 results:\n")

# Check if we found any significant advantages
found_advantage = False

# Check Exp 1 results
if len(df_exp1) > 0:
    for terrain in df_exp1['terrain'].unique():
        terrain_data = df_exp1[df_exp1['terrain'] == terrain]
        if len(terrain_data) > 0:
            spline_avg = terrain_data[terrain_data['activation'] == 'spline']['r2'].mean()
            relu_avg = terrain_data[terrain_data['activation'] == 'relu']['r2'].mean()
            adv_pct = 100 * (spline_avg - relu_avg) / relu_avg
            if adv_pct > 1.0:
                found_advantage = True
                break

# Check Exp 2 results
if not found_advantage and len(df_exp2) > 0:
    for terrain in df_exp2['terrain'].unique():
        for sh_level in df_exp2['sh_level'].unique():
            subset = df_exp2[(df_exp2['terrain'] == terrain) & (df_exp2['sh_level'] == sh_level)]
            if len(subset) > 0:
                spline_r2 = subset[subset['activation'] == 'spline']['r2'].values
                relu_r2 = subset[subset['activation'] == 'relu']['r2'].values
                if len(spline_r2) > 0 and len(relu_r2) > 0:
                    adv_pct = 100 * (spline_r2[0] - relu_r2[0]) / relu_r2[0]
                    if adv_pct > 1.0:
                        found_advantage = True
                        break

if found_advantage:
    print("✅ FOUND ADVANTAGE >1% in at least one configuration")
    print("\n📋 RECOMMENDATION:")
    print("   1. Download SRTM 30m data for high-resolution analysis (Exp 3)")
    print("   2. Download ESA CCI land cover for boundary tasks (Exp 5)")
    print("   3. Complete Experiments 3, 4, 5 to fully characterize advantage")
    print("\n🎯 POTENTIAL PUBLICATION: 'When and Where Learned Activations Excel'")
else:
    print("❌ NO SIGNIFICANT ADVANTAGE found in any configuration")
    print("\n📋 RECOMMENDATION:")
    print("   1. CONSIDER STOPPING HERE - strong evidence SH+ReLU is optimal")
    print("   2. OR proceed with Exp 3-5 for completeness (comprehensive negative result)")
    print("   3. Focus on mechanistic analysis (NB21) - WHY don't splines help?")
    print("\n🎯 POTENTIAL PUBLICATION: 'SH Encoding Obviates Learned Activations'")

print("\n" + "="*80)

---
## Experiments 3-5: Placeholders for Phase 2

**To be completed after downloading:**
- SRTM 30m elevation data
- ESA CCI land cover data
- Optional: OpenStreetMap building footprints

See [NOTEBOOK20_DATA_SOURCES.md](NOTEBOOK20_DATA_SOURCES.md) for download instructions.

In [ ]:
print("="*80)
print("PLACEHOLDER: EXPERIMENTS 3-5")
print("="*80)

print("\n⏳ Experiment 3: Multi-Resolution Within Regions")
print("   Status: Requires SRTM 30m data download")
print("   Purpose: Test if fine-resolution (1km) shows spline advantages")

print("\n⏳ Experiment 4: Urban vs Rural Patterns")
print("   Status: Can use existing GPW population data")
print("   Purpose: Test if urban boundaries favor splines")

print("\n⏳ Experiment 5: Boundary-Rich Tasks")
print("   Status: Coastlines available, land cover requires download")
print("   Purpose: Test step functions and sharp edges")

print("\n📋 To continue with Phase 2:")
print("   1. Review Phase 1 results above")
print("   2. Follow data download instructions in NOTEBOOK20_DATA_SOURCES.md")
print("   3. Add experiment code in new cells below")

print("\n" + "="*80)

---
## Summary

**Phase 1 Complete**: Experiments 1 & 2 using existing ETOPO data

**Files Generated**:
- `exp1_continental_comparisons.csv` - 5 continents × 3 activations
- `exp2_sh_encoding_levels.csv` - 3 SH levels × 2 terrains × 3 activations

**Next Steps**: See analysis and recommendations above